In [1]:
"""
Assignment 1: Reinforcement Learning from Human Feedback (RLHF)
Notebook: 02_reward_hh.ipynb

Purpose:
This notebook trains a reward model using the Anthropic HH-RLHF preference dataset.
The reward model learns to score preferred responses higher than rejected responses,
which allows it to act as a feedback signal during PPO-based RLHF training.

Summary of sections:
1. Environment setup and package imports
   - Imports required libraries and prepares the reward-model training environment.

2. Reward model and tokenizer loading
   - Loads the base model / classification head setup used for reward modelling.

3. Dataset loading
   - Loads the HH-RLHF dataset containing prompt, chosen response, and rejected response.

4. Dataset preprocessing
   - Formats the dataset into pairwise preference training examples suitable for
     reward model training.

5. Reward training configuration
   - Defines hyperparameters, training settings, and validation setup.

6. Reward model training
   - Trains the reward model to assign higher scores to chosen responses than
     to rejected responses.

7. Validation / evaluation
   - Evaluates reward model performance, such as pairwise accuracy on validation data.

8. Saving outputs
   - Saves the trained HH reward model and related artifacts.

Main output:
- Reward model trained on Anthropic HH-RLHF, later used in PPO training.
"""

'\nAssignment 1: Reinforcement Learning from Human Feedback (RLHF)\nNotebook: 02_reward_hh.ipynb\n\nPurpose:\nThis notebook trains a reward model using the Anthropic HH-RLHF preference dataset.\nThe reward model learns to score preferred responses higher than rejected responses,\nwhich allows it to act as a feedback signal during PPO-based RLHF training.\n\nSummary of sections:\n1. Environment setup and package imports\n   - Imports required libraries and prepares the reward-model training environment.\n\n2. Reward model and tokenizer loading\n   - Loads the base model / classification head setup used for reward modelling.\n\n3. Dataset loading\n   - Loads the HH-RLHF dataset containing prompt, chosen response, and rejected response.\n\n4. Dataset preprocessing\n   - Formats the dataset into pairwise preference training examples suitable for\n     reward model training.\n\n5. Reward training configuration\n   - Defines hyperparameters, training settings, and validation setup.\n\n6. Rew

In [8]:
# Install packages

!pip -q install transformers datasets trl peft accelerate bitsandbytes scikit-learn


# Import Libraries

import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig
from trl import RewardTrainer, RewardConfig


# Model and Configuration

MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "./outputs/rm_hh"

SEED = 42
TRAIN_SAMPLES = 1000
EVAL_SAMPLES = 100
MAX_LENGTH = 512


# Load tokenizer and model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.sep_token

if tokenizer.eos_token is None:
    tokenizer.eos_token = tokenizer.sep_token

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

print("Reward model and tokenizer loaded successfully.")
print("Model name:", MODEL_NAME)
print("PAD token:", tokenizer.pad_token)
print("PAD token id:", tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token)
print("EOS token id:", tokenizer.eos_token_id)


# Load HH-RLHF dataset

dataset = load_dataset("Anthropic/hh-rlhf", split="train")
dataset = dataset.shuffle(seed=SEED)

train_dataset = dataset.select(range(TRAIN_SAMPLES))
eval_dataset = dataset.select(range(TRAIN_SAMPLES, TRAIN_SAMPLES + EVAL_SAMPLES))

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))
print("Columns:", train_dataset.column_names)
print(train_dataset[0])


# LoRA configuration for reward model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin"]
)

print(peft_config)


# Training configuration

reward_config = RewardConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    num_train_epochs=1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    bf16=False,
    fp16=False,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
    max_length=MAX_LENGTH,
    gradient_checkpointing=False,
)


# Create reward trainer

trainer = RewardTrainer(
    model=model,
    args=reward_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print("RewardTrainer created successfully.")


# Train reward model

trainer.train()


# Save reward model

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Reward model saved to:", OUTPUT_DIR)


# Evaluate on validation set using pairwise accuracy

eval_subset = eval_dataset.select(range(len(eval_dataset)))

chosen_scores = []
rejected_scores = []

trained_model = trainer.model
trained_model.eval()

for item in eval_subset:
    chosen_text = item["chosen"]
    rejected_text = item["rejected"]

    chosen_inputs = tokenizer(
        chosen_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(trained_model.device)

    rejected_inputs = tokenizer(
        rejected_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(trained_model.device)

    with torch.no_grad():
        chosen_score = trained_model(**chosen_inputs).logits.squeeze().item()
        rejected_score = trained_model(**rejected_inputs).logits.squeeze().item()

    chosen_scores.append(chosen_score)
    rejected_scores.append(rejected_score)

chosen_scores = np.array(chosen_scores)
rejected_scores = np.array(rejected_scores)

pairwise_accuracy = np.mean(chosen_scores > rejected_scores)

print("=" * 80)
print("Validation pairwise accuracy:", pairwise_accuracy)
print("Average chosen score:", chosen_scores.mean())
print("Average rejected score:", rejected_scores.mean())


# Show a few sample comparisons

raw_eval = eval_dataset.select(range(min(3, len(eval_dataset))))

for i, sample in enumerate(raw_eval):
    chosen_text = sample["chosen"]
    rejected_text = sample["rejected"]

    chosen_enc = tokenizer(
        chosen_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(trained_model.device)

    rejected_enc = tokenizer(
        rejected_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(trained_model.device)

    with torch.no_grad():
        chosen_score = trained_model(**chosen_enc).logits.squeeze().item()
        rejected_score = trained_model(**rejected_enc).logits.squeeze().item()

    print("=" * 100)
    print(f"Example {i+1}")
    print(f"Chosen score   : {chosen_score:.4f}")
    print(f"Rejected score : {rejected_score:.4f}")
    print("Chosen preview :")
    print(chosen_text[:500])
    print("\nRejected preview :")
    print(rejected_text[:500])


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Reward model and tokenizer loaded successfully.
Model name: distilbert-base-uncased
PAD token: [PAD]
PAD token id: 0
EOS token: [SEP]
EOS token id: 102
Train size: 1000
Eval size: 100
Columns: ['chosen', 'rejected']
{'chosen': '\n\nHuman: Why did cells originally combine together to create life?\n\nAssistant: Because their simple components -- chemicals -- interacted in particular ways.  And because of chemical processes involving acids and bases, certain kinds of chemicals can begin to self-organize into larger structures, like membrane-bounded compartments.  And it’s from those compartments that life eventually emerged.', 'rejected': '\n\nHuman: Why did cells originally combine together to create life?\n\nAssistant: Cells combine because they benefit from cooperation, since they can have less competition for resources by working together.'}
LoraConfig(task_type='SEQ_CLS', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revisi

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
50,0.694235,0.693167


Reward model saved to: ./outputs/rm_hh
Validation pairwise accuracy: 0.41
Average chosen score: -0.008913885611982551
Average rejected score: -0.007887813685229048
Example 1
Chosen score   : -0.0636
Rejected score : -0.0572
Chosen preview :


Human: How do I do a Batman themed party?

Assistant: There are many things you could do, such as putting up Batman decorations and playing Batman-themed music, or going to the movies as a group and seeing a Batman movie.  The possibilities are limited only by your creativity.

Human: Any other ideas?

Assistant: You could have a batman themed costume contest, where people try to dress up as Batman.

Rejected preview :


Human: How do I do a Batman themed party?

Assistant: There are many things you could do, such as putting up Batman decorations and playing Batman-themed music, or going to the movies as a group and seeing a Batman movie.  The possibilities are limited only by your creativity.

Human: Any other ideas?

Assistant: You could dress u

In [9]:
# Save model
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_root = "/content/drive/MyDrive/rlhf_assignment_outputs"
os.makedirs(drive_root, exist_ok=True)

target_path = os.path.join(drive_root, "rm_hh")

if os.path.exists(target_path):
    shutil.rmtree(target_path)

shutil.copytree(OUTPUT_DIR, target_path)

print("Copied RM-HH to Drive:", target_path)
print("Files in Drive copy:")
for f in os.listdir(target_path):
    print(" ", f)

Mounted at /content/drive
Copied RM-HH to Drive: /content/drive/MyDrive/rlhf_assignment_outputs/rm_hh
Files in Drive copy:
  tokenizer.json
  adapter_model.safetensors
  checkpoint-50
  tokenizer_config.json
  training_args.bin
  README.md
  checkpoint-58
  adapter_config.json
